In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import torch.nn as nn
import pandas as pd

from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [2]:
# ============================================================================
# PROVEN OPTIMAL COMPONENTS (DO NOT CHANGE)
# ============================================================================

class RNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, 64, 1, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(64, 64)
        self.out = nn.Linear(64, 1)
        self.act = nn.ReLU()
    
    def forward(self, x):
        x, _ = self.rnn(x)
        return self.out(self.act(self.fc(x[:, -1, :])))


class WeightedL1(nn.Module):
    """PROVEN OPTIMAL: slope=1.0, threshold=2.0"""
    def forward(self, pred, target):
        weights = 1.0 + 1.0 * torch.clamp(torch.abs(target) - 2.0, min=0)
        return (torch.abs(pred - target) * weights).mean()


def predict(model, loader):
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(x.to(device)).cpu().numpy() for x, _ in loader]).flatten()

print("✅ Components ready")

✅ Components ready


In [3]:
# Load data
print("Loading data...")
train_loader, val_loader, pipeline = load_data(
    train_path="../datasets_challenge/train",
    train_ratio=0.98,
    batchsize=64,
    columns_to_drop=["sideSlip", "vyCG", "time", "longitude", "latitude"],
    sequence_length=20,
    threshold=True,
    threshold_value=3.0,
    threshold_column="vxCG",
    target="sideSlip",
    scaling=True,
    seed=42,
)

test_loader = load_test_data(
    test_path="../datasets_challenge/test",
    batchsize=512,
    pipeline=pipeline,
    load_anonymized_data=True
)

X_sample, _ = next(iter(train_loader))
input_size = X_sample.shape[2]
print(f"✅ Data loaded | Input size: {input_size}")

Loading data...
../datasets_challenge/train/train_part_001.csv
../datasets_challenge/train/train_part_002.csv
../datasets_challenge/train/train_part_003.csv
../datasets_challenge/train/train_part_004.csv
../datasets_challenge/train/train_part_005.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_012.csv
../datasets_challenge/train/train_part_012.csv
../datasets_challenge/train/train_part_013.csv
../datasets_challenge/train/train_part_014.csv
../datasets_challenge/train/train_part_015.c

In [4]:
def train_two_stage(seed, s2_lr, s2_epochs):
    """
    PROVEN OPTIMAL:
    - Stage 1: L1, lr=0.0003, 30 epochs (FIXED)
    - Stage 2: WeightedL1 slope=1.0 thresh=2.0 (FIXED), variable lr/epochs
    """
    seed_everything(seed)
    model = RNN(input_size).to(device)
    
    # Stage 1: Standard L1 (FIXED)
    print(f"  S1: L1, lr=0.0003, 30ep", end="")
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state = np.inf, None
    
    for ep in range(30):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    
    model.load_state_dict(state)
    print(f" → {best:.5f}")
    
    # Stage 2: Weighted L1 (VARIABLE lr/epochs)
    print(f"  S2: WeightedL1, lr={s2_lr}, {s2_epochs}ep", end="")
    opt = torch.optim.Adam(model.parameters(), lr=s2_lr)
    crit2 = WeightedL1()
    best2, state2 = np.inf, None
    
    for ep in range(s2_epochs):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit2(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best2: best2, state2 = val, model.state_dict().copy()
    
    model.load_state_dict(state2)
    print(f" → {best2:.5f}")
    return model, best2


def train_baseline(seed):
    seed_everything(seed)
    model = RNN(input_size).to(device)
    print(f"  Baseline L1", end="")
    
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state, patience = np.inf, None, 0
    
    for ep in range(50):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: 
            best, state, patience = val, model.state_dict().copy(), 0
        else:
            patience += 1
            if patience > 10: break
    
    model.load_state_dict(state)
    print(f" → {best:.5f}")
    return model, best

print("✅ Training functions ready")

✅ Training functions ready


## 🎯 Final Push: More Epochs at Optimal LR

In [5]:
# PROVEN OPTIMAL
SEEDS = [42, 2024]

# Final configs - push epochs at optimal LR
CONFIGS = [
    # (name, lr, epochs)
    ("lr5e5_45ep", 0.00005, 45),   # +5 epochs from best
    ("lr5e5_50ep", 0.00005, 50),   # +10 epochs from best
    ("lr5e5_55ep", 0.00005, 55),   # +15 epochs (find limit)
    ("lr4e5_45ep", 0.00004, 45),   # Slightly lower LR test
]

print("="*60)
print("🎯 FINAL PUSH: More Epochs at Optimal LR")
print(f"   Current best: lr=0.00005, 40ep → 0.21751")
print("="*60)

🎯 FINAL PUSH: More Epochs at Optimal LR
   Current best: lr=0.00005, 40ep → 0.21751


In [6]:
# Train baseline
print("\n📦 BASELINE")
baseline_preds = []
for seed in SEEDS:
    print(f"🌱 Seed {seed}")
    model, _ = train_baseline(seed)
    baseline_preds.append(predict(model, test_loader))
baseline_ensemble = np.mean(baseline_preds, axis=0)
print("✅ Baseline ready")


📦 BASELINE
🌱 Seed 42
  Baseline L1 → 0.11366
 → 0.11366
🌱 Seed 2024
  Baseline L1🌱 Seed 2024
  Baseline L1 → 0.11157
 → 0.11157
✅ Baseline ready
✅ Baseline ready


In [7]:
# Train all configs
results = {}

for name, lr, epochs in CONFIGS:
    print(f"\n{'='*50}")
    print(f"📦 {name} (lr={lr}, {epochs}ep)")
    print("="*50)
    
    preds = []
    for seed in SEEDS:
        print(f"🌱 Seed {seed}")
        model, _ = train_two_stage(seed, lr, epochs)
        preds.append(predict(model, test_loader))
    
    results[name] = np.mean(preds, axis=0)
    print(f"✅ {name} done")


📦 lr5e5_45ep (lr=5e-05, 45ep)
🌱 Seed 42
  S1: L1, lr=0.0003, 30ep → 0.12387
  S2: WeightedL1, lr=5e-05, 45ep → 0.12387
  S2: WeightedL1, lr=5e-05, 45ep → 0.11176
 → 0.11176
🌱 Seed 2024
  S1: L1, lr=0.0003, 30ep🌱 Seed 2024
  S1: L1, lr=0.0003, 30ep → 0.12372
  S2: WeightedL1, lr=5e-05, 45ep → 0.12372
  S2: WeightedL1, lr=5e-05, 45ep → 0.11139
 → 0.11139
✅ lr5e5_45ep done

📦 lr5e5_50ep (lr=5e-05, 50ep)
🌱 Seed 42
  S1: L1, lr=0.0003, 30ep✅ lr5e5_45ep done

📦 lr5e5_50ep (lr=5e-05, 50ep)
🌱 Seed 42
  S1: L1, lr=0.0003, 30ep → 0.12630
  S2: WeightedL1, lr=5e-05, 50ep → 0.12630
  S2: WeightedL1, lr=5e-05, 50ep → 0.11222
 → 0.11222
🌱 Seed 2024
  S1: L1, lr=0.0003, 30ep🌱 Seed 2024
  S1: L1, lr=0.0003, 30ep → 0.12364
  S2: WeightedL1, lr=5e-05, 50ep → 0.12364
  S2: WeightedL1, lr=5e-05, 50ep → 0.11187
 → 0.11187
✅ lr5e5_50ep done

📦 lr5e5_55ep (lr=5e-05, 55ep)
🌱 Seed 42
  S1: L1, lr=0.0003, 30ep✅ lr5e5_50ep done

📦 lr5e5_55ep (lr=5e-05, 55ep)
🌱 Seed 42
  S1: L1, lr=0.0003, 30ep → 0.12333
  S2: W

In [8]:
# Save 50/50 blends
print("\n" + "="*60)
print("💾 SAVING (50/50 blend with baseline)")
print("="*60)

for name, ens in results.items():
    blend = (baseline_ensemble + ens) / 2
    filename = f"pred_v8_{name}.csv"
    pd.DataFrame({"id": range(len(blend)), "predicted_sideslip": blend}).to_csv(filename, index=False)
    print(f"✅ {filename}")


💾 SAVING (50/50 blend with baseline)
✅ pred_v8_lr5e5_45ep.csv
✅ pred_v8_lr5e5_50ep.csv
✅ pred_v8_lr5e5_55ep.csv
✅ pred_v8_lr4e5_45ep.csv
✅ pred_v8_lr4e5_45ep.csv


In [9]:
# Also try blending best configs together
print("\n" + "="*60)
print("🔀 MULTI-CONFIG BLENDS")
print("="*60)

# Average of all new configs
all_new = np.mean([results[k] for k in results.keys()], axis=0)
blend_all = (baseline_ensemble + all_new) / 2
pd.DataFrame({"id": range(len(blend_all)), "predicted_sideslip": blend_all}).to_csv(
    "pred_v8_blend_all_new.csv", index=False)
print("✅ pred_v8_blend_all_new.csv (avg of all new configs)")

# Best 2 (45ep and 50ep at lr=5e-5)
if "lr5e5_45ep" in results and "lr5e5_50ep" in results:
    best2 = (results["lr5e5_45ep"] + results["lr5e5_50ep"]) / 2
    blend_best2 = (baseline_ensemble + best2) / 2
    pd.DataFrame({"id": range(len(blend_best2)), "predicted_sideslip": blend_best2}).to_csv(
        "pred_v8_blend_45_50ep.csv", index=False)
    print("✅ pred_v8_blend_45_50ep.csv (45ep + 50ep avg)")


🔀 MULTI-CONFIG BLENDS
✅ pred_v8_blend_all_new.csv (avg of all new configs)
✅ pred_v8_blend_45_50ep.csv (45ep + 50ep avg)


In [10]:
print("\n" + "="*60)
print("📋 FINAL SUMMARY")
print("="*60)

print("\n🏆 Current Best: 0.21751 (lr=0.00005, 40ep)")

print("\n📁 Files to submit:")
print("  Priority 1 (most likely to beat 0.21751):")
print("    - pred_v8_lr5e5_45ep.csv")
print("    - pred_v8_lr5e5_50ep.csv")
print("")
print("  Priority 2 (explore limits):")
print("    - pred_v8_lr5e5_55ep.csv")
print("    - pred_v8_lr4e5_45ep.csv")
print("")
print("  Priority 3 (ensemble diversity):")
print("    - pred_v8_blend_45_50ep.csv")
print("    - pred_v8_blend_all_new.csv")

print("\n💡 Hypothesis:")
print("  - 45ep or 50ep should beat 40ep (trend continues)")
print("  - 55ep might overfit (find the limit)")
print("  - lr=0.00004 likely worse (0.00005 proven optimal)")


📋 FINAL SUMMARY

🏆 Current Best: 0.21751 (lr=0.00005, 40ep)

📁 Files to submit:
  Priority 1 (most likely to beat 0.21751):
    - pred_v8_lr5e5_45ep.csv
    - pred_v8_lr5e5_50ep.csv

  Priority 2 (explore limits):
    - pred_v8_lr5e5_55ep.csv
    - pred_v8_lr4e5_45ep.csv

  Priority 3 (ensemble diversity):
    - pred_v8_blend_45_50ep.csv
    - pred_v8_blend_all_new.csv

💡 Hypothesis:
  - 45ep or 50ep should beat 40ep (trend continues)
  - 55ep might overfit (find the limit)
  - lr=0.00004 likely worse (0.00005 proven optimal)
